# ARC Prize 2026 — Entrenamiento LoRA (kernel de Kaggle, GPU L4×4)

Corre en GPU de Kaggle **L4×4 (96GB, internet OFF obligatorio)**. Pipeline: construir dataset
augmentado → fine-tune SFT con LoRA → guardar adaptador en `/kaggle/working` (luego se publica
como dataset para el kernel de inferencia `submit.ipynb`).

**Adjuntar a este kernel (todo offline):**
- La competencia `arc-prize-2026-arc-agi-2` (datos).
- El código del repo como dataset (con `src/` y `scripts/`).
- El **modelo base** desde Kaggle Models (p.ej. Qwen2.5-7B-Instruct) — NO se descarga de HF.

> L4×4 exige internet deshabilitado y consume quota al doble; solo disponible adjuntado a esta competencia.

In [ ]:
# OFFLINE (L4x4 exige internet OFF): nada de pip install ni descargas HF.
# La imagen GPU de Kaggle trae torch/transformers/peft/accelerate (NO bitsandbytes!).
import os, sys, glob, json, zipfile
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # anti-fragmentacion

OUT_DIR = '/kaggle/working/lora-adapter'
DATA_JSONL = '/kaggle/working/finetune_train.jsonl'

# RUN REAL: Qwen2.5-7B en fp16 repartido en los 2 T4 (sin 4-bit: no hay bitsandbytes offline).
SMOKE = False
N_AUG = 2 if SMOKE else 4            # augmentaciones (D4+color) por tarea REAL
N_SYNTH = 200 if SMOKE else 8000     # nº de tareas SINTÉTICAS
LOAD_4BIT = False                    # bitsandbytes NO esta en la imagen -> nada de 4-bit
DEVICE_MAP = 'cuda' if SMOKE else 'balanced'   # 7B fp16 repartido en los 2 T4
EPOCHS = 1.0
BATCH = 1
GRAD_ACCUM = 16
MAX_SEQ_LEN = 1024
MAX_STEPS = 40 if SMOKE else 1500   # cota de tiempo (1500*16 ~= 24k muestras vistas)
SAVE_STEPS = 300                     # checkpoints periodicos (respaldo si se corta)

print('INPUTS:', glob.glob('/kaggle/input/*'))

# Localiza el codigo (src/arc) en cualquier nivel; si viene como zips, los extrae
arc_init = glob.glob('/kaggle/input/**/src/arc/__init__.py', recursive=True)
if not arc_init:
    dst = '/kaggle/working/code'; os.makedirs(dst, exist_ok=True)
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        if os.path.basename(z) in ('src.zip', 'scripts.zip'):
            with zipfile.ZipFile(z) as zf:
                zf.extractall(os.path.join(dst, os.path.basename(z)[:-4]))
    arc_init = glob.glob(dst + '/**/arc/__init__.py', recursive=True)
assert arc_init, 'No se encontro src/arc. Adjunta el dataset de codigo (arc-agi2-code).'
SRC_DIR = os.path.dirname(os.path.dirname(arc_init[0]))
sys.path.insert(0, SRC_DIR)
sc = glob.glob('/kaggle/**/scripts/build_finetune_dataset.py', recursive=True)
if sc:
    sys.path.insert(0, os.path.dirname(sc[0]))

# Localiza los datos de la competencia (busca el json de training en cualquier ruta)
ch = glob.glob('/kaggle/input/**/arc-agi_training_challenges.json', recursive=True)
assert ch, 'No se encontraron los datos de la competencia. Adjunta arc-prize-2026-arc-agi-2.'
COMP_DIR = os.path.dirname(ch[0])

# MODELO BASE: dir con config.json + pesos (safetensors/bin) y SIN adapter_config.json
MODEL_DIR = None
for cfg in glob.glob('/kaggle/input/**/config.json', recursive=True):
    d = os.path.dirname(cfg)
    files = os.listdir(d)
    has_weights = any(f.endswith(('.safetensors', '.bin')) for f in files)
    if has_weights and 'adapter_config.json' not in files:
        MODEL_DIR = d; break
assert MODEL_DIR, 'Adjunta el modelo base (Kaggle Models) — p.ej. Qwen2.5-7B-Instruct.'
print('SMOKE =', SMOKE, '| device_map =', DEVICE_MAP, '| SRC_DIR =', SRC_DIR)
print('COMP_DIR =', COMP_DIR); print('MODEL_DIR =', MODEL_DIR)

In [ ]:
# 1) Construir dataset = tareas REALES augmentadas + tareas SINTÉTICAS
from arc.data import load_split
from arc.synth import generate_dataset
from build_finetune_dataset import samples_for_task

n = 0
with open(DATA_JSONL, 'w', encoding='utf-8') as f:
    # reales (D4 + color + leave-one-out)
    for task in load_split('training', data_dir=COMP_DIR).values():
        for s in samples_for_task(task, n_aug=N_AUG, max_cells=900, sep=''):
            f.write(json.dumps(s) + '\n'); n += 1
    # sintéticas (la regla es consistente por tarea; el modelo aprende a inducirla)
    for task in generate_dataset(N_SYNTH, seed=0):
        for s in samples_for_task(task, n_aug=1, max_cells=900, sep=''):
            f.write(json.dumps(s) + '\n'); n += 1
print('Muestras totales:', n, '->', DATA_JSONL)

In [ ]:
# 2) Fine-tune SFT con LoRA. Modelo base desde MODEL_DIR (offline, fp16, multi-GPU).
import torch
from transformers import Trainer, TrainingArguments
from arc.model import load_model, build_lora
from train_lora import ArcSFTDataset, PadCollator, read_jsonl

cap = torch.cuda.get_device_capability(0)[0]
BF16 = cap >= 8                       # Ampere+ (L4/A100). T4(7)/P100(6) -> fp16
print('GPU:', torch.cuda.get_device_name(0), '| n_gpus', torch.cuda.device_count(),
      '| cap', cap, '| bf16', BF16)

rows = list(read_jsonl(DATA_JSONL))
print('muestras:', len(rows))
model, tok = load_model(MODEL_DIR, load_4bit=LOAD_4BIT, device_map=DEVICE_MAP,
                        dtype='bfloat16' if BF16 else 'float16')
model.config.use_cache = False                      # requerido con gradient checkpointing
model = build_lora(model, r=16)
model.print_trainable_parameters()
# si el modelo quedo repartido en varias GPUs, marca model-parallel para el Trainer
if getattr(model, 'hf_device_map', None) and len(set(model.hf_device_map.values())) > 1:
    model.is_parallelizable = True; model.model_parallel = True

ds = ArcSFTDataset(rows, tok, max_seq_len=MAX_SEQ_LEN)
targs = TrainingArguments(
    output_dir='/kaggle/working/ckpt', num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=1e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
    logging_steps=20, save_strategy='steps', save_steps=SAVE_STEPS, save_total_limit=1,
    bf16=BF16, fp16=not BF16, optim='adamw_torch', report_to='none', ddp_find_unused_parameters=False)
trainer = Trainer(model=model, args=targs, train_dataset=ds,
                  data_collator=PadCollator(pad_id=tok.pad_token_id))
trainer.train()

In [ ]:
# 3) Guardar adaptador LoRA + tokenizer (sera la salida del kernel)
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print('LoRA guardado en', OUT_DIR)
print(os.listdir(OUT_DIR))

## Tras ejecutar
1. Guarda la salida del kernel (`/kaggle/working/lora-adapter`) como **Dataset** (p.ej. `arc-agi2-model`).
2. Adjunta ese dataset (y el modelo base) al kernel de inferencia `submit.ipynb` (internet OFF).
3. Valida en `evaluation` con `scripts/evaluate.py` antes de enviar.